In [1]:
import getpass, json, os, socket, sys
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

try:   # use the computer's own trusted certificates (fixes CERTIFICATE_VERIFY_FAILED on many Windows setups)
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    print("Tip: if you see CERTIFICATE_VERIFY_FAILED, run  pip install truststore  and restart the kernel.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))

def setting(name):
    v = os.getenv(name, "").strip()
    return "" if v.startswith("<") else v     # ignore a leftover <placeholder>

SIGNIN_URL = setting("ZETARIS_SIGNIN_URL")
API_BASE = setting("ZETARIS_REST_URL") or SIGNIN_URL
EMAIL = setting("ZETARIS_EMAIL")
PASSWORD = os.getenv("ZETARIS_PASSWORD") or getpass.getpass("Zetaris password: ")

def hide(text, secrets=(PASSWORD,)):
    # never print a password or token by accident
    text = str(text)
    for s in secrets:
        if s and len(s) > 3:
            text = text.replace(s, "***")
    return text

print("project folder :", ROOT)
print("sign-in URL    :", SIGNIN_URL or "(missing: add ZETARIS_SIGNIN_URL to .env)")
print("API base       :", API_BASE or "(missing)")
print("email          :", EMAIL or "(missing: add ZETARIS_EMAIL to .env)")
print("password set   :", bool(PASSWORD))

project folder : f:\Zeus AI\Bus-Tracker\bus-buddy-upload
sign-in URL    : https://new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com/login
API base       : https://new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com/login
email          : ujwalb016@gmail.com
password set   : True


In [4]:
## 2. Can we reach it?

`200`, `302` or `401` all mean the server answered. A timeout or `ConnectionError` means the address, the port or the cluster's network rules are wrong (or the cluster is stopped).

`CERTIFICATE_VERIFY_FAILED` means Python does not trust the site's certificate yet: run `pip install truststore`, restart the kernel and run again. Do not turn certificate checks off.


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
for name, url in [("sign-in", SIGNIN_URL), ("API base", API_BASE)]:
    if not url:
        continue
    u = urlparse(url)
    port = u.port or (443 if u.scheme == "https" else 80)
    try:
        socket.create_connection((u.hostname, port), timeout=10).close()
        print(f"{name}: port {port} open on {u.hostname}")
    except OSError as e:
        print(f"{name}: cannot open port {port} on {u.hostname}: {e}")
    try:
        r = requests.get(url, timeout=20, allow_redirects=False)
        print(f"   GET {url} -> {r.status_code} {r.headers.get('content-type', '')}")
    except Exception as e:
        print(f"   GET {url} failed: {e.__class__.__name__}: {e}")

sign-in: port 443 open on new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com
   GET https://new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com/login failed: SSLError: HTTPSConnectionPool(host='new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com', port=443): Max retries exceeded with url: /login (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
API base: port 443 open on new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com
   GET https://new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com/login failed: SSLError: HTTPSConnectionPool(host='new-ui.c2eb7cd9-30be-4f16-8dd2-d76a7182de2f.cloud.zetaris.com', port=443): Max retries exceeded with url: /login (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))


In [ ]:
CANDIDATES = ["/v3/api-docs", "/v2/api-docs", "/api-docs", "/swagger.json", "/openapi.json",
              "/api/v1.0/swagger.json", "/swagger-ui.html", "/swagger-ui/index.html", "/docs"]
found = []
for path in CANDIDATES:
    url = urljoin(API_BASE.rstrip("/") + "/", path.lstrip("/"))
    try:
        r = requests.get(url, timeout=15)
    except Exception as e:
        print(f"{path:<26} failed ({e.__class__.__name__})")
        continue
    print(f"{path:<26} {r.status_code} {r.headers.get('content-type', '')[:40]}")
    if r.ok and "json" in r.headers.get("content-type", ""):
        found.append(r.json())

for spec in found:
    paths = spec.get("paths", {})
    hits = [p for p in paths if any(w in p.lower() for w in ("login", "auth", "token", "query", "sql", "statement", "result"))]
    print(f"\nAPI description found: {len(paths)} endpoints, {len(hits)} look relevant:")
    for p in hits:
        print("  ", ", ".join(m.upper() for m in paths[p]), p)

In [ ]:
# ---- fill these in from step 3 (examples only, not the real paths) ----
LOGIN_PATH = ""        # e.g. "/api/v1.0/login"
LOGIN_BODY = {"email": EMAIL, "password": PASSWORD}   # change the field names to what the login request uses
QUERY_PATH = ""        # e.g. "/api/v1.0/query"  (the call that starts a SQL query)
# --------------------------------------------------------------------------

session = requests.Session()

def call(method, path, **kw):
    url = urljoin(API_BASE.rstrip("/") + "/", path.lstrip("/"))
    r = session.request(method, url, timeout=60, **kw)
    print(method, url, "->", r.status_code)
    print(hide(r.text[:600]))
    return r

if LOGIN_PATH:
    r = call("POST", LOGIN_PATH, json=LOGIN_BODY)
    body = r.json() if "json" in r.headers.get("content-type", "") else {}
    token = next((body[k] for k in ("token", "access_token", "accessToken", "jwt") if k in body), None)
    if token:
        session.headers["Authorization"] = f"Bearer {token}"
        print("logged in; token kept in the session (not printed)")
    else:
        print("no token field in the answer; the session may use a cookie instead:", list(session.cookies.keys()))
else:
    print("Fill in LOGIN_PATH first (step 3).")

In [ ]:
SQL = "SELECT * FROM NY_TAXI.taxi_zones LIMIT 5"

if QUERY_PATH:
    r = call("POST", QUERY_PATH, json={"sql": SQL})   # change "sql" to the field name the query request uses
else:
    print("Fill in QUERY_PATH first (step 3).")

If the answer holds the rows directly, turn them into a table with the next cell. If it holds a **query id** instead, add the
"fetch results" call you saw in step 3 (and the "close query" call) using `call(...)`, the same way as above.

In [ ]:
def to_frame(body):
    # common shapes: {"columns": [...], "rows": [[...]]}, {"data": [{...}]}, or a plain list of rows
    if isinstance(body, list):
        return pd.DataFrame(body)
    if "columns" in body and ("rows" in body or "data" in body):
        cols = [c["name"] if isinstance(c, dict) else c for c in body["columns"]]
        return pd.DataFrame(body.get("rows") or body.get("data"), columns=cols)
    for key in ("data", "rows", "results", "records"):
        if isinstance(body.get(key), list):
            return pd.DataFrame(body[key])
    raise ValueError(f"unknown answer shape, keys: {list(body)}")

if QUERY_PATH and r.ok:
    df = to_frame(r.json())
    display(df)

## 5. Our data: Zetaris vs the local files

Once our tables are registered in Zetaris, put their names below. The same question is asked both ways; the rows should match.
`app.adapters.zetaris.Zetaris(sample_mode=True)` is exactly what the app uses today (DuckDB over `data/sample/`).

In [ ]:
ZETARIS_TABLE = ""   # e.g. "busbuddy.maintenance_reports" (the name Zetaris shows after registering)
LOCAL_SQL = "SELECT bus_id, reported_at, issue, severity FROM maintenance_reports WHERE service_date = '2025-04-29' ORDER BY bus_id"

from app.adapters.zetaris import Zetaris
local_rows, _ = Zetaris(sample_mode=True).query(LOCAL_SQL)
print("local files:")
display(pd.DataFrame(local_rows))

if ZETARIS_TABLE and QUERY_PATH:
    r = call("POST", QUERY_PATH, json={"sql": LOCAL_SQL.replace("maintenance_reports", ZETARIS_TABLE)})
    print("Zetaris:")
    display(to_frame(r.json()))